# 01 — Mandi Price Data: Acquisition & Inspection

**Workstream:** Price Prediction · **Batch 1 (Day 1-2)** · **Status: IMPLEMENTED**

**Purpose:** get mandi price data into the repo in the confirmed real schema, validate it, and do a first
structural inspection — before any cleaning, EDA, or feature engineering (those are notebook 02+).

**Prerequisites:**
- Repo root on `sys.path` (handled in the next cell)
- Optional: `DATA_GOV_IN_API_KEY` in `.env` for a full real pull. **Not required to run this notebook** — 
  without it, this notebook uses a clearly-labelled synthetic fixture with the exact same schema, so the
  pipeline can be built and tested this week regardless.

**Data source used by this run:** decided automatically by `ai/price_prediction/src/ingest.load_or_fetch()` —
prints which one below. A small (10-row) **real** sample from data.gov.in is also cross-checked against the
same schema, from a live fetch done during Batch 1 (12 Aug 2026).

**Next notebook:** `02_eda_cleaning.ipynb` (Batch 2) — takes the output of this notebook's "Save snapshot" cell.


## 1. Setup

In [1]:
import sys
from pathlib import Path

# Make the repo root importable regardless of where Jupyter's cwd lands.
# Walk upward until we find the directory that actually contains ai/common/config.py,
# instead of assuming a fixed number of parent hops (fragile if the notebook moves).
_candidate = Path.cwd()
_root = None
for _p in [_candidate, *_candidate.parents]:
    if (_p / "ai" / "common" / "config.py").exists():
        _root = _p
        break
if _root is None:
    raise RuntimeError("Could not locate repo root (ai/common/config.py not found in any parent).")
sys.path.insert(0, str(_root))
print("Repo root on sys.path:", _root)


Repo root on sys.path: /home/claude/AgriConnect


In [2]:
import pandas as pd
from ai.common.config import DATA_GOV_IN_API_KEY, DATA_RAW_DIR, MANDI_RESOURCE_ID
from ai.common.validation import validate_mandi_schema
from ai.common.io_utils import save_df
from ai.price_prediction.src.ingest import load_or_fetch, fetch_mandi_prices_sample

print("MANDI_RESOURCE_ID:", MANDI_RESOURCE_ID)
print("Registered DATA_GOV_IN_API_KEY present:", bool(DATA_GOV_IN_API_KEY))


MANDI_RESOURCE_ID: 9ef84268-d588-465a-a308-a864a43d0070
Registered DATA_GOV_IN_API_KEY present: False


## 2. Load data (real if key is present, else synthetic fixture)

`load_or_fetch()` always prefers the most-real option available:
`real_full` (your registered key) > `synthetic` (fixture, this run, since no key is set yet).

In [3]:
df, source = load_or_fetch()
print(f"Loaded {len(df):,} rows from source = '{source}'")
if source == "synthetic":
    print("NOTE: this is SYNTHETIC data for pipeline development only — see docs/architecture/decision-log.md.")
df.head()


08:25:06 | WARNING  | ai.price_prediction.src.ingest | Generated 500 SYNTHETIC mandi price rows - NOT real data. Do not use for reported metrics.


08:25:06 | INFO     | ai.price_prediction.src.ingest | Real 10-row seed sample also available at /home/claude/AgriConnect/data/raw/mandi_prices/seed_sample_REAL_10rows_2026-08-11.csv for schema cross-check.


Loaded 500 rows from source = 'synthetic'
NOTE: this is SYNTHETIC data for pipeline development only — see docs/architecture/decision-log.md.


,state,district,market,commodity,variety,grade,arrival_date,min_price,max_price,modal_price,source,extracted_at,is_synthetic
0,Tamil Nadu,Tamil District,Chennai Mandi,Rice,Standard,FAQ,06/03/2026,2468.69,3253.10,2860.90,SYNTHETIC_FIXTURE,2026-08-12T08:25:06.883672+00:00,True
1,Maharashtra,Maharashtra District,Nashik Mandi,Tomato,Standard,FAQ,07/05/2026,1135.87,1367.77,1251.82,SYNTHETIC_FIXTURE,2026-08-12T08:25:06.883672+00:00,True
2,Maharashtra,Maharashtra District,Pune Mandi,Wheat,Standard,FAQ,28/03/2026,2174.06,2444.71,2309.38,SYNTHETIC_FIXTURE,2026-08-12T08:25:06.883672+00:00,True
3,Uttar Pradesh,Uttar District,Agra Mandi,Potato,Standard,FAQ,15/06/2026,898.12,1076.01,987.07,SYNTHETIC_FIXTURE,2026-08-12T08:25:06.883672+00:00,True
4,Tamil Nadu,Tamil District,Coimbatore Mandi,Tomato,Standard,FAQ,26/06/2026,1073.93,1309.43,1191.68,SYNTHETIC_FIXTURE,2026-08-12T08:25:06.883672+00:00,True


## 3. Cross-check against the REAL seed sample

A genuinely real 10-row sample was pulled live from data.gov.in during Batch 1 using the provider's own
public test key (hard-capped at 10 rows/call — not a training set, just a schema/connectivity proof).

In [4]:
real_seed_path = DATA_RAW_DIR / "mandi_prices" / "seed_sample_REAL_10rows_2026-08-11.csv"
real_seed = pd.read_csv(real_seed_path)
print(f"Real seed sample: {len(real_seed)} rows, fetched live 12 Aug 2026")
real_seed


Real seed sample: 10 rows, fetched live 12 Aug 2026


,state,district,market,commodity,variety,grade,arrival_date,min_price,max_price,modal_price,source,extracted_at
0,Odisha,Mayurbhanja,Baripada APMC,Cucumbar(Kheera),Cucumbar,Medium,11/08/2026,3000,3500,3200,data.gov.in,2026-08-12T00:00:00Z
1,Odisha,Mayurbhanja,Baripada APMC,Brinjal,Brinjal,Medium,11/08/2026,4500,6000,5000,data.gov.in,2026-08-12T00:00:00Z
2,Andhra Pradesh,Prakasam,Maddipadu APMC,Bengal Gram(Gram)(Whole),Gulabi,FAQ,11/08/2026,5900,5900,5900,data.gov.in,2026-08-12T00:00:00Z
3,Tamil Nadu,Thirupur,Tiruppur (South) (Uzhavar Sandhai ),Onion,Bellary,Local,11/08/2026,3000,3500,3250,data.gov.in,2026-08-12T00:00:00Z
4,Tamil Nadu,Thirupur,Tiruppur (South) (Uzhavar Sandhai ),Cluster beans,Cluster Beans,Local,11/08/2026,3000,4000,3500,data.gov.in,2026-08-12T00:00:00Z
5,Tamil Nadu,Thirupur,Tiruppur (South) (Uzhavar Sandhai ),Bottle gourd,Bottle Gourd,Local,11/08/2026,1500,2000,1750,data.gov.in,2026-08-12T00:00:00Z
6,Tamil Nadu,Thirupur,Tiruppur (South) (Uzhavar Sandhai ),Amaranthus,Amaranthus,Local,11/08/2026,2500,3000,2750,data.gov.in,2026-08-12T00:00:00Z
7,Tamil Nadu,Thirupur,Tiruppur (South) (Uzhavar Sandhai ),Carrot,Pusakesar,Local,11/08/2026,7000,8000,7500,data.gov.in,2026-08-12T00:00:00Z
8,Tamil Nadu,Thirupur,Tiruppur (South) (Uzhavar Sandhai ),Snakeguard,Snakeguard,Local,11/08/2026,3000,4000,3500,data.gov.in,2026-08-12T00:00:00Z
9,Tamil Nadu,Thirupur,Tiruppur (South) (Uzhavar Sandhai ),Lemon,Lemon,Local,11/08/2026,7000,8000,7500,data.gov.in,2026-08-12T00:00:00Z


## 4. Schema validation (both the working dataset and the real seed sample)

In [5]:
working_result = validate_mandi_schema(df)
seed_result = validate_mandi_schema(real_seed)
print("Working dataset :", working_result.report())
print("Real seed sample:", seed_result.report())
assert working_result.ok, "Fix schema issues before proceeding to notebook 02."
assert seed_result.ok, "Real seed sample should always be schema-valid — investigate if not."


Working dataset : VALID - no schema issues found.
Real seed sample: VALID - no schema issues found.


## 5. Structural inspection

In [6]:
print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)
print("\nDate range (raw arrival_date, DD/MM/YYYY strings):",
      df["arrival_date"].min(), "to", df["arrival_date"].max())
print("\nUnique states   :", df["state"].nunique(), sorted(df["state"].unique()))
print("Unique markets  :", df["market"].nunique())
print("Unique commodities:", df["commodity"].nunique(), sorted(df["commodity"].unique()))


Shape: (500, 13)

Dtypes:
 state               str
district            str
market              str
commodity           str
variety             str
grade               str
arrival_date        str
min_price       float64
max_price       float64
modal_price     float64
source              str
extracted_at        str
is_synthetic       bool
dtype: object

Date range (raw arrival_date, DD/MM/YYYY strings): 01/03/2026 to 31/07/2026

Unique states   : 5 ['Andhra Pradesh', 'Karnataka', 'Maharashtra', 'Tamil Nadu', 'Uttar Pradesh']
Unique markets  : 10
Unique commodities: 5 ['Onion', 'Potato', 'Rice', 'Tomato', 'Wheat']


## 6. Missing values & duplicates

In [7]:
print("Missing values per column:\n", df.isna().sum())
dup_cols = ["state", "market", "commodity", "variety", "arrival_date"]
dup_count = df.duplicated(subset=dup_cols).sum()
print(f"\nDuplicate rows on {dup_cols}: {dup_count}")

zero_price_rows = df[(df["modal_price"] <= 0)]
print(f"Rows with modal_price <= 0 (should be 0 — rule: missing price is NaN, never 0): {len(zero_price_rows)}")


Missing values per column:
 state           0
district        0
market          0
commodity       0
variety         0
grade           0
arrival_date    0
min_price       0
max_price       0
modal_price     0
source          0
extracted_at    0
is_synthetic    0
dtype: int64

Duplicate rows on ['state', 'market', 'commodity', 'variety', 'arrival_date']: 0
Rows with modal_price <= 0 (should be 0 — rule: missing price is NaN, never 0): 0


## 7. Save inspected snapshot

Feeds `02_eda_cleaning.ipynb`. Saved to `data/processed/`, never overwriting `data/raw/`
(leakage rule 1, `docs/price/target-and-leakage.md`).

In [8]:
from ai.common.config import DATA_PROCESSED_DIR

out_path = DATA_PROCESSED_DIR / f"mandi_prices_inspected_{source}.csv"
save_df(df, out_path)
print(f"Saved {len(df):,} rows ({source}) -> {out_path}")


Saved 500 rows (synthetic) -> /home/claude/AgriConnect/data/processed/mandi_prices_inspected_synthetic.csv


## Next steps

- **If `source == 'synthetic'`:** get a registered `DATA_GOV_IN_API_KEY` (see `PROGRESS_LEDGER.md` →
  ACTION REQUIRED) and re-run this notebook — `load_or_fetch()` will automatically switch to `real_full`
  once the key is in `.env`. No code changes needed.
- **If `source == 'real_full'`:** proceed straight to `02_eda_cleaning.ipynb`.
- Either way, the commodity/market shortlist in `docs/price/target-and-leakage.md` gets finalized against
  whatever `df["commodity"].value_counts()` / `df.groupby(["state","market","commodity"]).size()` actually
  show — not assumed in advance (Knowledge 04: "validate list in Week 1").

**Status of this notebook: IMPLEMENTED.** Everything above actually ran (see `PROGRESS_LEDGER.md` for the
Batch 1 execution log) — nothing here is a description of what it will do.